# LLM で文章を要約する (長さと観点を指定する)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/23_summarization/llm_summary.ipynb)

NLP クックブック 23-04 ／ 要約と平易化

文章の内容を、自分の言葉でまとめ直した要約 (生成型要約) が欲しい。長さや、何に注目するかを指定したい。CPU で動かしたい。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # 3B の日本語モデル。CPU でも 1 要約あたり十数秒 (メモリ約 7GB)
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

def chat(prompt, max_new_tokens=150):
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

text = ("市は 14 日、来年 4 月から市内の全小中学校で給食費を無償化すると発表した。対象は約 1 万 2 千人で、年間の費用は約 6 億円を見込む。"
        "財源には、ふるさと納税の寄付金と、公共施設の統廃合で生じる削減分を充てる。"
        "市長は記者会見で「子育て世帯の負担を減らし、若い世代に選ばれるまちにしたい」と述べた。"
        "一方、市議会の一部からは「恒久的な財源の裏付けが弱い」との指摘が出ている。"
        "市は来月の議会に関連する条例案を提出する方針で、可決されれば県内の市では 3 例目となる。"
        "保護者からは歓迎の声が上がる一方、食材費の高騰で給食の質が下がらないか心配する意見もある。")
print(len(text), "文字")
print(chat(f"次の文章を要約してください。\n\n{text}"))

In [ ]:
# 長さを指定する
for instruction in ["1 文 (40 字以内) で要約してください。", "100 字程度で要約してください。"]:
    summary = chat(f"次の文章を{instruction}要約だけを出力してください。\n\n{text}")
    print(f"[{instruction}] ({len(summary)} 字) {summary}")

In [ ]:
# 観点や読み手を指定する
for focus in ["財源と費用に注目して、2 文で要約してください。", "賛成と反対の意見を 1 文ずつにまとめてください。", "小学生にもわかる言葉で、2 文で要約してください。"]:
    print(f"[{focus}]\n   {chat(focus + chr(10) + chr(10) + text)}")

- 生成型要約は、文をつなぎ直したり言い換えたりできるので、抽出型より短く自然にまとまる。そのかわり、原文にないことを書く (ハルシネーション) おそれがある。数値と固有名詞は必ず原文と照らし合わせる。
- 文字数の指定は目安にしかならない。厳密な上限が必要なら、生成後に文字数を確かめて、超えていたら「もっと短く」と再生成する。
- 「誰のための、何のための要約か」を書くと質が上がる。同じ文章でも、目的が違えば残すべき情報が違う。
- 小さいモデルは、長い入力の後半を見落としやすい。長い文書は「長い文書を分割して要約する」のレシピを参照。